## About
- This notebook includes training and evaluation of all two-feature combinations using symbolic regression

### Input
> All single feature and two-feature combination models
>
> performance metrics across training, test and validation sets of the above models

### Output
> Top 10 models based on AUROC in the test set
>
> Visualziations of model performance including hotspot proteins

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
from sklearn.metrics import roc_auc_score, roc_curve
from src.plots import plot_roc_curve

In [ ]:
# Define project directory
Base = 'data_directory'

In [ ]:
# Define project directory
df_olink = pd.read_excel(os.path.join(Base, 'data_curated/cureated_proteomics_clinical_data.xlsx'), sheet_name='proteomics_curated').set_index('CBMRID')
df_cli = pd.read_excel(os.path.join(Base, 'data_curated/cureated_proteomics_clinical_data.xlsx'), sheet_name='clinical_curated').set_index('CBMRID')

# Join curated proteomics and clinical data
data = df_olink.join(df_cli)
# Drop samples without outcome target measure
data = data[data['cohort']!='GALA_LiHep'].dropna(subset=['inflam_binary'])

# import MS data 
data_ms = pd.read_csv(os.path.join(Base, 'data_curated/olink_ms_combined.csv')).set_index('CBMRID')
data=data.join(data_ms[['Q08380']])

In [ ]:
# Read ROC-AUC metrics
roc_curves_combined_file = os.path.join(Base, 'symbolic_regression/models/roc_curves_combined.pkl')
with open(roc_curves_combined_file , 'rb') as f:
    roc_curves_combined = pickle.load(f)

In [ ]:
# Read final models 
df_final = pd.read_excel('supp_tables/ST6.xlsx', sheet_name='ST6')
topn = df_final[df_final['top7']]['features'].unique()

#### Sensitivity analysis for I3 and I4, same predicted score but different labels, use ROC-AUC

In [ ]:
# Import labels
with open(os.path.join(Base, 'symbolic_regression/splits.pkl'), 'rb') as f:
    cohort_index_dict = pickle.load(f)

In [ ]:
def plot_roc_curve(true_y, y_prob, color):
    """
    plots the roc curve based of the probabilities
    """

    fpr, tpr, thresholds = roc_curve(true_y, y_prob)
    plt.plot(fpr, tpr, color)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')

In [ ]:
best_combo = ('CDCP1', 'ast_log2')
#best_combo = ('HGF', 'TWEAK')
#best_combo = ('HGF', 'TRAIL')
#best_combo = ('Q08380','SCF')
roc_curves_best = roc_curves_combined[best_combo]
roc_curves_ast = roc_curves_combined['ast_log2']

for outcome in ['inflam_binary_I3', 'inflam_binary_I4']:    
    for set_toplot in ['test', 'validation1']:
        y_true = data.loc[cohort_index_dict[set_toplot]][outcome]        
        y_prob_best = roc_curves_best[set_toplot][1]
        y_prob_ast = roc_curves_ast[set_toplot][1]
        
        fig, axs = plt.subplots(figsize=(4,4))
        plot_roc_curve(y_true, y_prob_best, color='steelblue')
        plot_roc_curve(y_true, y_prob_ast, color='gray')
        
        auroc_best = roc_auc_score(y_true, y_prob_best)
        auroc_ast = roc_auc_score(y_true, y_prob_ast)
        
        plt.plot([0, 1], [0, 1], ls="--", c='lightgray')
        plt.rcParams['pdf.fonttype'] = 42
        plt.title(set_toplot+'_'+outcome)
        plt.text(0.4, 0.2, 'ast auroc:{}'.format(round(auroc_ast, 2)), color='gray')
        plt.text(0.4, 0.1, '{}+{} auroc:{}'.format(best_combo[0], best_combo[1], round(auroc_best, 2)), color='steelblue')
        # plt.savefig(os.path.join(Base, 'symbolic_regression/figures/sensitivity_analysis_{}_{}.png'.format(set_toplot, outcome)), 
        #            dpi=120, bbox_inches='tight')

#### Sensitivity analysis seperatly for NAFLD and ALD risks

In [ ]:
ald_risk_ids = data[(data['aldrisk']==1) | (data['cohort']=='HP')].index.tolist() 
nafld_risk_ids = data[data['cohort'].isin(['RDC', 'SIP', 'HP'])].index.tolist()

In [ ]:
data['cohort'].value_counts()

In [ ]:
data[data['cohort'].isin(['RDC', 'HP', 'ALD'])]['aldrisk'].value_counts(1)

In [ ]:
data.loc[nafld_risk_ids]['inflam_binary'].value_counts(1)

In [ ]:
data.loc[ald_risk_ids]['inflam_binary'].value_counts(1)

In [ ]:
outcome = 'I2'
roc_curves_best = roc_curves_combined[best_combo]

for set_toplot in ['test', 'validation1']:
    y_true = roc_curves_best[set_toplot][0]
    y_prob = pd.Series(index=y_true.index, data=roc_curves_best[set_toplot][1])   
    
    y_true_nafld = y_true.loc[y_true.index.isin(nafld_risk_ids)]
    y_prob_nafld = y_prob.loc[y_prob.index.isin(nafld_risk_ids)]
    y_true_ald = y_true.loc[y_true.index.isin(ald_risk_ids)]
    y_prob_ald = y_prob.loc[y_prob.index.isin(ald_risk_ids)]
    
    fig, axs = plt.subplots(figsize=(4,4))
    plot_roc_curve(y_true_nafld, y_prob_nafld, color='steelblue')
    auroc_nafld = roc_auc_score(y_true_nafld, y_prob_nafld)
    if set_toplot=='test':
        plot_roc_curve(y_true_ald, y_prob_ald, color='red')
        auroc_ald = roc_auc_score(y_true_ald, y_prob_ald)
        class1_pert = int(round(y_true_ald.sum()/len(y_true_ald)*100, 0))
        plt.text(0.3, 0.1, 'ALD auroc:{}, case={}%'.format(round(auroc_ald, 2), class1_pert), color='red')
    else:
        pass
    plt.plot([0, 1], [0, 1], ls="--", c='lightgray')
    plt.rcParams['pdf.fonttype'] = 42
    plt.title(set_toplot+'_'+outcome)
    class1_pert_nafld = int(round(y_true_nafld.sum()/len(y_true_nafld)*100, 0))
    
    plt.text(0.3, 0.2, 'MASLD auroc:{}, case={}%'.format(round(auroc_nafld, 2), class1_pert_nafld), color='steelblue')
    # plt.savefig(os.path.join(Base, 'symbolic_regression/figures/sensitivity_analysis_etiology_{}.png'.format(set_toplot)), 
    #            dpi=120, bbox_inches='tight')

### Joint figure

In [ ]:
best_combo = ('SCF', 'SLAMF1')
roc_curves_best = roc_curves_combined[best_combo]
roc_curves_ast = roc_curves_combined['ast_log2']

fig, axes = plt.subplots(3, 2, figsize=(8, 12))
plt.subplots_adjust(hspace=0.1)

# --- Panels a, b: NAFLD vs ALD (I2) ---
outcome = 'I2'
for col, set_toplot in enumerate(['test', 'validation1']):
    ax = axes[0, col]
    y_true = roc_curves_best[set_toplot][0]
    y_prob = pd.Series(index=y_true.index, data=roc_curves_best[set_toplot][1])

    y_true_nafld = y_true.loc[y_true.index.isin(nafld_risk_ids)]
    y_prob_nafld = y_prob.loc[y_prob.index.isin(nafld_risk_ids)]
    y_true_ald = y_true.loc[y_true.index.isin(ald_risk_ids)]
    y_prob_ald = y_prob.loc[y_prob.index.isin(ald_risk_ids)]

    plt.sca(ax)
    plot_roc_curve(y_true_nafld, y_prob_nafld, color='steelblue')
    auroc_nafld = roc_auc_score(y_true_nafld, y_prob_nafld)
    ax.text(0.3, 0.2, 'MASLD AUROC:{}, n={}'.format(round(auroc_nafld, 2), len(y_true_nafld)), color='steelblue')

    if set_toplot == 'test':
        plot_roc_curve(y_true_ald, y_prob_ald, color='red')
        auroc_ald = roc_auc_score(y_true_ald, y_prob_ald)
        ax.text(0.3, 0.1, 'ALD AUROC:{}, n={}'.format(round(auroc_ald, 2), len(y_true_ald)), color='red')

    ax.plot([0, 1], [0, 1], ls="--", c='lightgray')
    ax.set_title('MASLD vs. ALD {}'.format(set_toplot))
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')

# --- Panels c-f: I3 and I4 ---
for row, outcome in enumerate(['inflam_binary_I3', 'inflam_binary_I4']):
    for col, set_toplot in enumerate(['test', 'validation1']):
        ax = axes[row + 1, col]
        y_true = data.loc[cohort_index_dict[set_toplot]][outcome]
        y_prob_best = roc_curves_best[set_toplot][1]
        y_prob_ast = roc_curves_ast[set_toplot][1]

        auroc_best = roc_auc_score(y_true, y_prob_best)
        auroc_ast = roc_auc_score(y_true, y_prob_ast)

        plt.sca(ax)
        plot_roc_curve(y_true, y_prob_best, color='steelblue')
        plot_roc_curve(y_true, y_prob_ast, color='gray')
        ax.plot([0, 1], [0, 1], ls="--", c='lightgray')
        ax.text(0.3, 0.2, 'AST AUROC:{}'.format(round(auroc_ast, 2)), color='gray')
        ax.text(0.3, 0.1, '{}+{} AUROC:{}'.format(best_combo[0], best_combo[1], round(auroc_best, 2)), color='steelblue')

        label = outcome.replace('inflam_binary_', '')
        title = '{} set {}'.format('Validation cohort' if set_toplot == 'validation1' else 'Test set', label)
        ax.set_title(title)
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')

plt.rcParams['pdf.fonttype'] = 42
plt.tight_layout()
for ax, label in zip(axes.flat, 'abcdef'):
    ax.text(-0.25, 1.2, label , transform=ax.transAxes, fontsize=12, fontweight='bold', va='top')
#plt.savefig(os.path.join(Base, 'symbolic_regression/figures/panel_figure.png'), dpi=120, bbox_inches='tight')
plt.savefig('panel_figure_{}.pdf'.format(best_combo), bbox_inches='tight')